In [1]:
print("hai")

hai


In [3]:
%pip install langchain-anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 8.9 MB/s eta 0:00:00 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepeval 2.6.7 requires anthropic<0.50.0,>=0.49.0, but you have anthropic 0.96.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import operator
import math
import datetime
from typing import TypedDict, Annotated, Literal
from dotenv import load_dotenv

# ── Claude via LangChain ──────────────────────────────────────────────────
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMessage,
)
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# Load ANTHROPIC_API_KEY from .env
load_dotenv(override=True)
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
if not anthropic_api_key:
    raise ValueError("ANTHROPIC_API_KEY not found. Add it to your .env file.")

# ── Initialise Claude chat model ──────────────────────────────────────────
chat_model = ChatAnthropic(
    model="claude-opus-4-6",
    temperature=0.3,
    anthropic_api_key=anthropic_api_key,
)

from importlib.metadata import version
print("✅  Setup complete!")
print(f"   LangGraph       : {version('langgraph')}")
print(f"   langchain-anthropic : {version('langchain-anthropic')}")
print(f"   Model           : claude-opus-4-6")

# Quick sanity check — confirm Claude responds
_test = chat_model.invoke([HumanMessage(content="Say hello in one word.")])
print(f"   Test call       : {_test.content.strip()}")

✅  Setup complete!
   LangGraph       : 1.0.1
   langchain-anthropic : 0.3.0
   Model           : claude-opus-4-6
   Test call       : Hello!


In [5]:
import requests
import json
import os
from dotenv import load_dotenv

# ─── Load environment variables from .env file ────────────────────────────────
load_dotenv(override=True)  # override=True allows .env values to overwrite existing env vars

# ─── Configuration ────────────────────────────────────────────────────────────
API_KEY = os.getenv("ANTHROPIC_API_KEY")
ENDPOINT = "https://api.anthropic.com/v1/messages"

# ─── Headers (same as Postman Headers tab) ────────────────────────────────────
headers = {
    "x-api-key": API_KEY,
    "anthropic-version": "2023-06-01",
    "content-type": "application/json"
}

# ─── Request Body (same as Postman Body → raw → JSON) ─────────────────────────
payload = {
    "model": "claude-sonnet-4-6",
    "max_tokens": 256,
    "temperature": 0.5,
    "system": "You are a helpful assistant.",
    "messages": [
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ]
}

# ─── Send POST Request ─────────────────────────────────────────────────────────
response = requests.post(ENDPOINT, headers=headers, json=payload)

# ─── Display Response ──────────────────────────────────────────────────────────
print(f"Status Code : {response.status_code}")
print(f"Response    :\n{json.dumps(response.json(), indent=2)}")

# ─── Extract just the assistant's reply ───────────────────────────────────────
if response.status_code == 200:
    reply = response.json()["content"][0]["text"]
    print(f"\nClaude says: {reply}")


Status Code : 200
Response    :
{
  "model": "claude-sonnet-4-6",
  "id": "msg_01QKjU2svDxLHK33GSr15Fwg",
  "type": "message",
  "role": "assistant",
  "content": [
    {
      "type": "text",
      "text": "The capital of France is **Paris**."
    }
  ],
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "stop_details": null,
  "usage": {
    "input_tokens": 21,
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "cache_creation": {
      "ephemeral_5m_input_tokens": 0,
      "ephemeral_1h_input_tokens": 0
    },
    "output_tokens": 11,
    "service_tier": "standard",
    "inference_geo": "global"
  }
}

Claude says: The capital of France is **Paris**.
